# lane_3_maxmix_scans: dependency-aware A100 queue

This Colab lane runs `04_maxmix_master → 05_scans_and_controls`. Run `00_validation` separately first; after it passes, open all three lane notebooks in the three available A100 sessions and run all cells. Every completed archive is receipt/checksum verified on restart, including compatible legacy S=25 supersets, so prior work is skipped rather than repeated. The lane checks shared Drive storage before every new shard and writes a resumable session record under `_lane_sessions`.

These lanes default to `production`. Production stops when an accepted-width or evidence-backed launch-gate file is absent; do not leave an A100 idling while waiting for a human gate decision.

## 1. Mount Drive and locate the complete campaign folder


In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

MYDRIVE = Path('/content/drive/MyDrive')
candidates = [
    MYDRIVE / 'final_production_ready_figure_scripts',
]
candidates.extend(path.parent for path in MYDRIVE.glob('*/colab_lane_runner.py'))
roots = []
for path in candidates:
    if (path / 'colab_lane_runner.py').is_file() and path not in roots:
        roots.append(path)
if len(roots) != 1:
    raise RuntimeError(f'expected one complete updated campaign folder; found {roots}')
CAMPAIGN_ROOT = roots[0]
print(f'[campaign] {CAMPAIGN_ROOT}')


## 2. Select the queue profile


In [ ]:
RUN_PROFILE = 'production'  # production, pilot_calibration, or pilot_science
LANE = 'lane_3_maxmix_scans'
MAX_SESSION_GPU_HOURS_OVERRIDE = None
PREFLIGHT_ONLY = False
RESUME_REPORT_ONLY = False
HEARTBEAT_SECONDS = 60
RUN_QUEUE = True
if RUN_PROFILE not in ('pilot_calibration', 'pilot_science', 'production'):
    raise ValueError(RUN_PROFILE)
if HEARTBEAT_SECONDS <= 0:
    raise ValueError('HEARTBEAT_SECONDS must be positive')
print(dict(lane=LANE, bundles=['04_maxmix_master', '05_scans_and_controls'], profile=RUN_PROFILE, preflight_only=PREFLIGHT_ONLY, resume_report_only=RESUME_REPORT_ONLY, heartbeat_seconds=HEARTBEAT_SECONDS))


## 3. Run the guarded lane


In [ ]:
if not RUN_QUEUE:
    raise RuntimeError('RUN_QUEUE is False')
command = [
    sys.executable, '-u', str(CAMPAIGN_ROOT / 'colab_lane_runner.py'),
    '--drive-root', str(MYDRIVE),
    '--lane', LANE,
    '--profile', RUN_PROFILE,
    '--heartbeat-seconds', str(HEARTBEAT_SECONDS),
]
if MAX_SESSION_GPU_HOURS_OVERRIDE is not None:
    command += ['--max-session-hours', str(MAX_SESSION_GPU_HOURS_OVERRIDE)]
if PREFLIGHT_ONLY:
    command.append('--preflight-only')
if RESUME_REPORT_ONLY:
    command.append('--resume-report-only')
runner_path = CAMPAIGN_ROOT / 'colab_lane_runner.py'
runner_build_id = hashlib.sha256(runner_path.read_bytes()).hexdigest()[:16]
collection = ('classA_final_production_outputs/production_10sample_v2' if RUN_PROFILE == 'production' else 'classA_pilot_outputs/production_10sample_v2')
output_root = MYDRIVE / collection
session_dir = output_root / '_lane_sessions'
print('[lane dashboard]')
print(json.dumps({
    'lane': LANE, 'bundles': ['04_maxmix_master', '05_scans_and_controls'], 'profile': RUN_PROFILE,
    'output_root': str(output_root), 'runner_build_id': runner_build_id,
    'heartbeat_seconds': HEARTBEAT_SECONDS, 'session_dir': str(session_dir),
}, indent=2))
print(f'[launch lane] {" ".join(command)}', flush=True)
result = subprocess.run(command, check=False)
if result.returncode:
    sessions = sorted(session_dir.glob(f'*_{LANE}_*.json'), key=lambda path: path.stat().st_mtime, reverse=True)
    if sessions:
        latest = json.loads(sessions[0].read_text(encoding='utf-8'))
        print('[saved lane failure]')
        print(json.dumps({
            'session_json': str(sessions[0]),
            'status': latest.get('status'),
            'current': latest.get('current'),
            'failure': latest.get('failure'),
            'session_log': latest.get('session_log'),
        }, indent=2))
    raise subprocess.CalledProcessError(result.returncode, command)


## Final. Disconnect this completed Colab runtime


In [ ]:
from google.colab import runtime
runtime.unassign()
